In [1]:
from dotenv import load_dotenv
load_dotenv()

True

# Research Agent

This one follows a simple, yet effective pattern

1. **LLM Decision Node**: Analyzes the current state and decides whether to make tool calls or provide a final response
2. **Tool Execution Node**: Executes search tools when the LLM determines more information is needed
3. **Research Compression Node**: Summarizes and compresses research findings for efficient processing
4. **Routing Logic**: Determines workflow continuation based on LLM decisions


### Design Considerations

#### 1. Prompt

First, we'll define a prompt that instructs our agent to use available search tools. A strong prompt is the heart of this agent. Think of the agent like a new employee, and you as the product manager.

**1. Think Like The Agent**

- **Read the question carefully** - What specific information does the user need?
- **Start with broader searches** - Use broad, comprehensive queries first
- **After each search, pause and assess** - Do I have enough to answer? What's still missing?
- **Execute narrower searches as you gather information** - Fill in the gaps.

**2. Budget control (Prevent "Spin-Out" on excessive tool calls)**
- **Stop when you can answer confidently** - Don't keep searching for perfection.
- **Give it budgets** - Use 2-3 search tool calls for simple queries. Use up to 5 for complex queries, or even a Time out.
- **Limit** - Always stop after 5 search tool calls if you cannot find the right source(s).

**3. Thinking abilities**
After each search tool calling, use [`think_tool`](https://www.anthropic.com/engineering/claude-think-tool) to analyze the results:
- What key information did I find? 
- What's missing?
- Do I have enough to answer the question comprehensively?
- Should I search more or provide my answer?

**Results**
These techniques transform problematic research behavior like:
```
"Best restaurants in Vancouver" → "Big Way Hot Pot review" → "Le Crocodile review" → etc. (10+ searches)
```

Into efficient patterns like:
```
"Best restaurants in Vancouver" → ThinkTool(analyze results) → "Restaurants with consistently high ratings" → ThinkTool(assess completeness) → provide answer (3-5 searches total)
```

#### 2. Context Engineering Strategy

We apply **context engineering** to process extensive information efficiently while maintaining high-quality research output using web page content summarization & research result compression.


**`summarize_webpage_content()` function**
- Uses structured output to extract key information and relevant excerpts
- Filters out irrelevant content (navigation, ads, boilerplate content) while preserving factual details
- Compresses lengthy articles into focused summaries
- Maintains source attribution for credibility

**`compress_research()` function**
- Synthesizes findings from multiple tool calls into cohesive insights
- Extracts raw notes for detailed analysis while maintaining compressed summaries
- Reduces token usage for subsequent LLM calls
- Preserves essential information for report writing

#### 3. Performing Careful Context Compression

Compress the overall context when it gets too long (human messages, long system prompts, tool call/results, etc.)


We need to be very careful about loosing valuable information. The long context can cause the compression LLM to loose sight of the task instructions (lost in the middle), leading to generic summaries that loose information. So, we reinforce the compression task by adding a `compress_research_human_message`.

- Explicitly restates the original research topic at compression time
- Reminds the model to preserve ALL information relevant to the specific question
- Emphasizes that comprehensive findings are critical for final report generation
- Prevents task drift during the compression phase

#### 4. Output Token Management
Research compression can generate long outputs. We need to sure that they do not exceed model token limits, which can cause truncated responses that cut off mid-sentence.

Setup `max_tokens` to ensure complete output. This prevents incomplete compression outputs and ensures full research findings are preserved.

## Agent Setup

### States & Schema

In [ ]:
# %%writefile ./src/state_research.py

"""
State Definitions and Pydantic Schemas for Research Agent

This module defines the state objects and structured schemas used for
the research agent workflow, including researcher state management and output schemas.
"""

import operator
from typing import TypedDict, Annotated, List, Sequence
from pydantic import BaseModel, Field
from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages

########## STATE DEFINITIONS ##########

class ResearcherState(TypedDict):
    """
    State for the research agent containing message history and research metadata.
    
    This state tracks the researcher's conversation, iteration count for limiting
    tool calls, the research topic being investigated, compressed findings,
    and raw research notes for detailed analysis.
    """
    researcher_messages: Annotated[Sequence[BaseMessage], add_messages] # A sequence of messages representing the conversation history of the researcher.
    tool_call_iterations: int
    research_topic: str
    compressed_research: str
    raw_notes: Annotated[List[str], operator.add]

class ResearcherOutputState(TypedDict):
    """
    Output state for the research agent containing final research results.
    
    This represents the final output of the research process with compressed
    research findings and all raw notes from the research process.
    """
    compressed_research: str
    raw_notes: Annotated[List[str], operator.add]
    researcher_messages: Annotated[Sequence[BaseMessage], add_messages]

########## OUTPUT SCHEMAS ##########

class ClarifyUserRequest(BaseModel): # From Scope Agent
    """Structured output for requesting clarification from the user."""
    need_clarification: Annotated[bool, Field(..., description="Indicates whether clarification is needed from the user.")] 
    question_to_user: Annotated[str, Field(default="", description="The question to ask the user for clarification. Required only when need_clarification=True; otherwise empty string.")]
    verification: Annotated[str, Field(default="", description="Acknowledgment message confirming the user's request is sufficient to proceed with research. Required only when need_clarification=False; otherwise empty string.")]

class ResearchBrief(BaseModel): # From Scope Agent
    """Structured output for the research brief generated from user input."""
    research_brief: Annotated[str, Field(..., description="A concise research brief summarizing the user's request and the research objectives to guide the research agents.")]

class WebSearchSummary(BaseModel):
    """Schema for webpage content summarization."""
    summary: str = Field(description="Concise summary of the webpage content")
    key_points: str = Field(description="Important quotes and excerpts from the content")

### Tool Setup

In [ ]:
# %%writefile ./src/agent_tools.py

from concurrent.futures import ThreadPoolExecutor
from typing import List, Literal, Annotated

from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool, InjectedToolArg
from tavily import TavilyClient

# custom imports
from src.agent_prompts import summarize_webpage_prompt
from src.utils import date_today_str

########## INITIALIZATION ##########

# Summarization is mechanical compression - Haiku is ~3x cheaper than Sonnet
# at >=90% quality on this task, and dominates per-tavily-search cost.
model = ChatAnthropic(model="claude-haiku-4-5", temperature=0)
tavily_client = TavilyClient() # for sequential web search, for parallel search use AsyncTavilyClient()

########### WEB SEARCH SETUP (Helper Functions for Tavily Search) ##########

def tavily_search_multiple(
    search_queries: List[str],
    max_results: int = 3,
    topic: Literal["general", "news", "finance"] = "general",
    include_raw_content: bool = True,
) -> List[dict]:
    """
    Perform search using Tavily API for multiple queries.

    Args:
        search_queries: List of search queries to execute
        max_results: Maximum number of results per query
        topic: Topic filter for search results
        include_raw_content: Whether to include raw webpage content

    Returns:
        List of search result dictionaries
    """

    search_docs = []
    for query in search_queries:
        result = tavily_client.search(
            query,
            max_results=max_results,
            include_raw_content=include_raw_content,
            topic=topic
        )
        search_docs.append(result)
    return search_docs


def summarize_webpage_content(webpage_content: str) -> str:
    """
    Summarize webpage content using the configured summarization model.

    Args:
        webpage_content: Raw webpage content to summarize

    Returns:
        Formatted summary with key excerpts
    """
    try:
        webpage_summarization_model = model.with_structured_output(WebSearchSummary)
        webpage_summarization_prompt = summarize_webpage_prompt.format(
            webpage_content=webpage_content,
            date=date_today_str()
        )
        summary = webpage_summarization_model.invoke([
            HumanMessage(content=webpage_summarization_prompt)
        ])

        formatted_summary = (
            f"<summary>\n{summary.summary}\n</summary>\n\n"
            f"<key_points>\n{summary.key_points}\n</key_points>"
        )

        return formatted_summary

    except Exception as e:
        print(f"Failed to summarize webpage: {str(e)}")
        return webpage_content[:1000] + "..." if len(webpage_content) > 1000 else webpage_content

def deduplicate_search_results(search_results: List[dict]) -> dict:
    """
    Deduplicate search results by URL to avoid processing duplicate content.

    Args:
        search_results: List of search result dictionaries

    Returns:
        Dictionary mapping URLs to unique results
    """
    unique_results = {}

    for response in search_results:
        for result in response['results']:
            url = result['url']
            if url not in unique_results:
                unique_results[url] = result
    return unique_results

def process_search_results(unique_results: dict) -> dict:
    """
    Process search results by summarizing content where available.

    Summarizations run in parallel via ThreadPoolExecutor: each raw_content
    page triggers one Anthropic LLM call, and waiting for them sequentially
    used to dominate every tavily_search invocation.

    Args:
        unique_results: Dictionary of unique search results

    Returns:
        Dictionary of processed results with summaries
    """
    if not unique_results:
        return {}

    items = list(unique_results.items())

    def process_one(item: tuple[str, dict]) -> tuple[str, dict]:
        url, result = item
        if not result.get("raw_content"):
            content = result['content']
        else:
            content = summarize_webpage_content(result['raw_content'])
        return url, {'title': result['title'], 'content': content}

    with ThreadPoolExecutor(max_workers=min(8, len(items))) as executor:
        processed = list(executor.map(process_one, items))

    return dict(processed)

def format_search_output(summarized_results: dict) -> str:
    """
    Format search results into a well-structured string output.

    Args:
        summarized_results: Dictionary of processed search results

    Returns:
        Formatted string of search results with clear source separation
    """
    if not summarized_results:
        return "No valid search results found. Please try different search queries or use a different search API."

    formatted_output = "Search results: \n\n"

    for i, (url, result) in enumerate(summarized_results.items(), 1):
        formatted_output += f"\n\n--- SOURCE {i}: {result['title']} ---\n"
        formatted_output += f"URL: {url}\n\n"
        formatted_output += f"SUMMARY:\n{result['content']}\n\n"
        formatted_output += "-" * 80 + "\n"

    return formatted_output

############### RESEARCH AGENT TOOLS ###############

# NOTE: `max_results` and `topic` are wrapped in `Annotated[..., InjectedToolArg]`.
# `InjectedToolArg` hides the argument from the tool schema sent to the LLM, so the
# model only sees `query` and cannot override our defaults at runtime. The values
# are still injected by LangGraph at execution time. Use this whenever a tool
# parameter is operator-controlled (budgets, runtime config, secrets) rather than
# something the LLM should choose.
@tool(parse_docstring=True)
def tavily_search(
    query: str,
    max_results: Annotated[int, InjectedToolArg] = 3,
    topic: Annotated[Literal["general", "news", "finance"], InjectedToolArg] = "general",
) -> str:
    """
    Fetch results from Tavily search API with content summarization.

    Args:
        query: A single search query to execute
        max_results: Maximum number of results to return
        topic: Topic to filter results by ('general', 'news', 'finance')

    Returns:
        Formatted string of search results with summaries
    """
    search_results = tavily_search_multiple(
        [query],  # Convert single query to list for the internal function
        max_results=max_results,
        topic=topic,
        include_raw_content=True,
    )

    unique_results = deduplicate_search_results(search_results) # Deduplicate results by URL to avoid processing duplicate content
    summarized_results = process_search_results(unique_results) # Process results with summarization
    return format_search_output(summarized_results)

@tool(parse_docstring=True)
def think_tool(reflection: str) -> str:
    """Tool for strategic reflection on research progress and decision-making.

    Use this tool after each search to analyze results and plan next steps systematically.
    This creates a deliberate pause in the research workflow for quality decision-making.

    When to use:
    - After receiving search results: What key information did I find?
    - Before deciding next steps: Do I have enough to answer comprehensively?
    - When assessing research gaps: What specific information am I still missing?
    - Before concluding research: Can I provide a complete answer now?

    Reflection should address:
    1. Analysis of current findings - What concrete information have I gathered?
    2. Gap assessment - What crucial information is still missing?
    3. Quality evaluation - Do I have sufficient evidence/examples for a good answer?
    4. Strategic decision - Should I continue searching or provide my answer?

    Args:
        reflection: Your detailed reflection on research progress, findings, gaps, and next steps

    Returns:
        Confirmation that reflection was recorded for decision-making
    """
    return f"Reflection recorded: {reflection}"

### Core Agent Setup

In [ ]:
# %%writefile ./src/research_agent.py

"""
Research Agent Core Implementation.

This module implements a research agent that can perform iterative web searches
and synthesis to answer complex research questions.
"""

from typing import Literal

from langgraph.graph import StateGraph, START, END
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage, filter_messages

# custom imports
from src.agent_prompts import research_agent_prompt, compress_research_system_prompt, compress_research_human_message

############ INITIALIZATION SETUP ##########

tools = [tavily_search, think_tool]
tools_by_name = {tool.name: tool for tool in tools}

model = ChatAnthropic(model="claude-sonnet-4-6", temperature=0)
model_with_tools = model.bind_tools(tools)
context_compression_model = ChatAnthropic(model="claude-sonnet-4-6", max_tokens=32000, temperature=0)

# Hard cap on tool-calling iterations to prevent runaway loops if the LLM
# ignores the prompt budget and keeps calling tavily_search.
MAX_TOOL_ITERATIONS = 5


########### RESEARCH AGENT NODES ##########

def llm_call(state: ResearcherState):
    """
    Analyze current state and decide on next actions.

    The model analyzes the current conversation state and decides whether to:
    1. Call search tools to gather more information
    2. Provide a final answer based on gathered information

    Returns updated state with the model's response.
    """

    research_agent_prompt_formatted = research_agent_prompt.format(date=date_today_str())
    return {
        "researcher_messages": [
            model_with_tools.invoke(
                [SystemMessage(content=research_agent_prompt_formatted)] + state["researcher_messages"]
            )
        ]
    }

def tool_node(state: ResearcherState):
    """
    Execute all tool calls from the previous LLM response.

    Executes all tool calls from the previous LLM responses.
    Returns updated state with tool execution results and incremented
    tool_call_iterations counter for loop-budget enforcement.
    """
    tool_calls = state["researcher_messages"][-1].tool_calls

    # Execute all tool calls
    observations = []
    for tool_call in tool_calls:
        tool = tools_by_name[tool_call["name"]]
        observations.append(tool.invoke(tool_call["args"]))

    # Create tool message outputs
    tool_outputs = [
        ToolMessage(
            content=observation,
            name=tool_call["name"],
            tool_call_id=tool_call["id"]
        ) for observation, tool_call in zip(observations, tool_calls)
    ]

    return {
        "researcher_messages": tool_outputs,
        "tool_call_iterations": state.get("tool_call_iterations", 0) + 1,
    }

def compress_research(state: ResearcherState) -> dict:
    """Compress research findings into a concise summary.

    Takes all the research messages and tool outputs and creates
    a compressed summary suitable for the supervisor's decision-making.
    """
    system_message = compress_research_system_prompt.format(date=date_today_str())
    human_message = compress_research_human_message.format(
        research_topic=state.get("research_topic", "")
    )
    messages = [SystemMessage(content=system_message)] + state.get("researcher_messages", []) + [HumanMessage(content=human_message)]
    response = context_compression_model.invoke(messages)

    # Extract raw notes from tool and AI messages
    raw_notes = [
        str(m.content) for m in filter_messages(
            state["researcher_messages"],
            include_types=["tool", "ai"]
        )
    ]

    return {
        "compressed_research": str(response.content),
        "raw_notes": ["\n".join(raw_notes)]
    }

# Conditional function to determine whether to continue research or provide final answer
def should_continue(state: ResearcherState) -> Literal["tool_node", "compress_research"]:
    """
    Determine whether to continue research or provide final answer.

    Stops when:
    1. The LLM stopped calling tools (no tool_calls on last message), OR
    2. The hard iteration cap was reached.

    Returns:
        "tool_node": Continue to tool execution
        "compress_research": Stop and compress research
    """
    last_message = state["researcher_messages"][-1]

    if not last_message.tool_calls:
        return "compress_research"

    if state.get("tool_call_iterations", 0) >= MAX_TOOL_ITERATIONS:
        return "compress_research"

    return "tool_node"


########### RESEARCH AGENT GRAPH ##########

research_agent_graph = StateGraph(ResearcherState, output_schema=ResearcherOutputState)

# Nodes
research_agent_graph.add_node("llm_call", llm_call)
research_agent_graph.add_node("tool_node", tool_node)
research_agent_graph.add_node("compress_research", compress_research)

# Edges
research_agent_graph.add_edge(START, "llm_call")
research_agent_graph.add_conditional_edges(
    "llm_call",
    should_continue,
    {
        "tool_node": "tool_node", # Continue research loop
        "compress_research": "compress_research", # Provide final answer
    },
)
research_agent_graph.add_edge("tool_node", "llm_call")
research_agent_graph.add_edge("compress_research", END)

research_agent = research_agent_graph.compile()

from IPython.display import Image, display
display(Image(research_agent.get_graph(xray=True).draw_mermaid_png()))

## Testing

In [ ]:
from src.utils import format_messages

# sample output from scope agent run
research_brief = """
I am learning about Agentic AI best practices for production environments. Please provide a comprehensive,         
educational deep-dive covering all of the following areas as they apply to real-world production deployments of    
agentic AI systems:                                                                                                

 1. System Design & Architecture - Best practices for designing agentic AI systems (e.g., agent orchestration,      
   multi-agent frameworks, tool use, memory management, context window handling, modularity, and scalability).     
 2. Safety, Alignment & Guardrails - How to implement effective guardrails, constrain agent behavior, prevent       
   unintended actions, ensure alignment with intended goals, and handle failure modes gracefully in production.    
 3. Security & Risk Management - Key security considerations such as prompt injection, data leakage, privilege      
   escalation, supply chain risks, and how to mitigate them in agentic pipelines.                                  
 4. Deployment & Operations - Production deployment strategies, monitoring, observability, logging, tracing agent   
   actions, debugging, latency management, cost control, and reliability engineering for agentic systems.          
 5. Governance & Ethics - Responsible AI practices, human-in-the-loop design, auditability, accountability,         
   regulatory considerations, and ethical frameworks relevant to deploying autonomous agents.                      

The research should be grounded in current (up to May 2026) industry knowledge, frameworks, and real-world         
production experience. Prioritize authoritative and primary sources such as official documentation from leading AI 
labs and frameworks (e.g., OpenAI, Anthropic, Google DeepMind, LangChain, AutoGen, CrewAI), peer-reviewed research,
and well-regarded engineering blogs (e.g., from companies like Databricks, Hugging Face, or similar). The audience 
is a learner seeking to understand and apply these best practices in real production settings, so clarity, concrete
examples, and actionable guidance are important.        
"""

response = research_agent.invoke(
    {
        "researcher_messages": [HumanMessage(content=research_brief)]
    }
)

format_messages(response["researcher_messages"])

Sample LangSmith Tracing is [here](https://smith.langchain.com/public/f407dbe2-334a-44f6-9856-3cbbe0f2260b/r)

In [7]:
from rich.markdown import Markdown
Markdown(response['compressed_research'])

Cleaned Research Findings: Agentic AI Best Practices for Production Environments                  

-------------------------------------------------------------------------------------------------------------------

List of Queries and Tool Calls Made                                                                                

 1 Query 1: "Agentic AI production best practices 2024 2025 system design architecture orchestration"              
 2 Query 2: "Agentic AI safety guardrails prompt injection security risks production 2024 2025"                    
 3 Query 3: "Agentic AI production deployment monitoring observability tracing LangSmith OpenTelemetry 2024 2025"  
 4 Query 4: "Agentic AI governance ethics human-in-the-loop accountability regulatory EU AI Act responsible AI     
   2025"                                                                                                           
 5 Query 5: "Multi-agent frameworks AutoGen CrewAI LangGraph production patterns cost control reliability 2025"    

-------------------------------------------------------------------------------------------------------------------

Fully Comprehensive Findings                                                                                       

-------------------------------------------------------------------------------------------------------------------

AREA 1: SYSTEM DESIGN & ARCHITECTURE                                                                               

Source: Anthropic — "Building Effective Agents" (December 20, 2024)                                                

Available at both anthropic.com/engineering/building-effective-agents and                                          
anthropic.com/research/building-effective-agents                                                                   

What are agents?                                                                                                   

▌ "In agentic contexts, LLMs operate with greater autonomy, execute long multi-step tasks, and work within larger
▌ systems involving multiple tools or agents, presenting unique challenges around performance and reliability    
▌ that don't exist in simpler LLM API usage."                                                                    

When to use agents:                                                                                                

▌ "Not every task requires an agent. Agents are best suited for tasks that benefit from LLM decision-making,     
▌ where the task is too complex or long for a single context window, or where the task benefits from having      
▌ multiple independent agents checking each other's work."                                                       

Building blocks: workflows vs. agents:                                                                             

▌ "We define workflows as systems where LLMs and tools are orchestrated through predefined, fixed code paths.    
▌ Agents, on the other hand, are systems where LLMs dynamically determine their own process and tool usage based 
▌ on user input."                                                                                                

When to use workflows vs. agents:                                                                                  

▌ "Workflows offer predictability and consistency for well-defined tasks, while agents are better for tasks that 
▌ require flexibility and model-driven decision-making at scale."                                                

Augmented LLMs:                                                                                                    

▌ "The basic building block of agentic systems is an LLM enhanced with augmentations such as retrieval, tools,   
▌ and memory. Our models can actively use these capabilities—searching for relevant information, maintaining     
▌ context across interactions, and taking actions—to complete complex tasks."   

# Agent Evaluation

Potential evaluation criteria:
- Early Termination: The agent decides to stop calling tools when the task is not actually complete. 
- Prolonged Looping: The agent is never satisfied with the state of its information. 

The first failure  mode results in the agent collecting insufficient information. In the deep research context, the answer may end up being too shallow. The second failure mode results in the agent utilizing an excessive number of tokens. In the deep research context, irrelevant information may poison the result.

#### Mock conversation:

Case-1: **Agent should CONTINUE - has insufficient/irrelevant information + think_tool realizes more research needed** (failure case)

In [8]:
from langchain_core.messages import AIMessage, ToolMessage, HumanMessage
messages_should_continue = [
    HumanMessage(content="What are the top coffee shops in San Francisco based on coffee quality?"),
    AIMessage(
        content="I'll search for information about top coffee shops in San Francisco focusing on coffee quality.",
        tool_calls=[
            {
                "name": "tavily_search",
                "args": {
                    "query": "coffee shops San Francisco popular locations"
                },
                "id": "call_continue_search_001",
                "type": "tool_call"
            }
        ]
    ),
    # Tavily Search provides generic cafe information - NOT focused on coffee quality
    ToolMessage(
        content="Search results show popular San Francisco cafes: Blue Bottle Coffee has great wifi and ambiance, Philz Coffee is known for custom blends and friendly service, Café Reveille has multiple locations with good pastries and laptop-friendly spaces. Ritual Coffee offers cozy seating and is popular with students.",
        tool_call_id="call_continue_search_001",
    ),
    # Agent uses think_tool to reflect and realizes the search results are insufficient
    AIMessage(
        content="Let me use the think_tool to analyze these search results and determine next steps.",
        tool_calls=[
            {
                "name": "think_tool",
                "args": {
                    "reflection": "Analysis of search results: The information I found focuses on general cafe features like wifi, ambiance, custom blends, and seating - but lacks specific coffee quality metrics. The user specifically asked for evaluation based on coffee quality, which would require expert reviews, professional ratings, specialty certifications, or quality-focused customer feedback. I need to search specifically for coffee quality assessments, Coffee Review scores, specialty coffee credentials, or expert evaluations of SF coffee shops. The current information is insufficient to answer the user's question about coffee quality."
                },
                "id": "call_continue_think_001",
                "type": "tool_call"
            }
        ]
    ),
    # think_tool response identifies the gap and need for more targeted research
    ToolMessage(
        content="Reflection recorded: Analysis of search results: The information I found focuses on general cafe features like wifi, ambiance, custom blends, and seating - but lacks specific coffee quality metrics. The user specifically asked for evaluation based on coffee quality, which would require expert reviews, professional ratings, specialty certifications, or quality-focused customer feedback. I need to search specifically for coffee quality assessments, Coffee Review scores, specialty coffee credentials, or expert evaluations of SF coffee shops. The current information is insufficient to answer the user's question about coffee quality.",
        tool_call_id="call_continue_think_001",
    )
]

Case-2: **Agent should STOP - has comprehensive coffee quality information + think_tool reflection** (good case)

In [9]:
messages_should_stop = [
    HumanMessage(content="What are the top coffee shops in San Francisco based on coffee quality?"),
    AIMessage(
        content="I'll research San Francisco coffee shops specifically focusing on coffee quality assessments and expert reviews.",
        tool_calls=[
            {
                "name": "tavily_search",
                "args": {
                    "query": "best coffee quality San Francisco specialty coffee expert reviews Coffee Review ratings 2024"
                },
                "id": "call_stop_search_001", 
                "type": "tool_call"
            }
        ]
    ),
    # This ToolMessage provides COMPREHENSIVE coffee quality information - sufficient to answer the question
    ToolMessage(
        content="""Coffee quality research for San Francisco reveals top establishments based on expert reviews and specialty coffee credentials:

**Top Quality-Focused Coffee Shops:**
1. **Sightglass Coffee** - Coffee Review score 94/100, known for single-origin beans and precise roasting techniques. Featured in Bon Appétit's "Best Coffee in America" list for exceptional coffee quality.

2. **Saint Frank Coffee** - Ranked #3 coffee shop in America by The Daily Meal based on coffee quality. 4.3/5 Yelp rating with reviews specifically praising "exceptional bean quality" and "perfect extraction."

3. **Ritual Coffee Roasters** - Specialty Coffee Association member, direct trade relationships, consistently rated 4.4/5 on Google with reviews highlighting "superior coffee taste" and "expert barista skills."

4. **Blue Bottle Coffee** - Third-wave coffee pioneer, Coffee Review scores averaging 90+/100, known for rigorous quality control and fresh roasting practices.

5. **Four Barrel Coffee** - Local roaster with Coffee Review scores 88-92/100, praised by industry experts for "exceptional coffee quality" and innovative brewing methods.

**Quality Indicators Found:**
- Coffee Review professional scores (88-94/100 range)
- Specialty Coffee Association certifications
- Direct trade/single-origin sourcing
- Expert recognition from food publications
- Customer reviews specifically mentioning coffee taste and quality

This provides a comprehensive assessment of SF coffee shops based specifically on coffee quality rather than ambiance or service.""",
        tool_call_id="call_stop_search_001",
    ),
    # Agent now uses think_tool to reflect on the search results
    AIMessage(
        content="Let me reflect on the search results using the think_tool to assess whether I have sufficient information.",
        tool_calls=[
            {
                "name": "think_tool",
                "args": {
                    "reflection": "Analysis of search results: I found comprehensive coffee quality information for SF coffee shops including: 1) Professional Coffee Review scores (88-94/100), 2) Expert rankings from reputable publications, 3) Specialty Coffee Association certifications, 4) Specific quality indicators (single-origin, roasting techniques, extraction methods), 5) Five top-rated establishments with detailed quality assessments. This covers all aspects requested: expert reviews, customer ratings, specialty certifications. I have enough information to provide a well-supported ranking based specifically on coffee quality. No additional searches needed - I should provide my comprehensive answer now."
                },
                "id": "call_think_001",
                "type": "tool_call"
            }
        ]
    ),
    # think_tool response confirming reflection was recorded
    ToolMessage(
        content="Reflection recorded: Analysis of search results: I found comprehensive coffee quality information for SF coffee shops including: 1) Professional Coffee Review scores (88-94/100), 2) Expert rankings from reputable publications, 3) Specialty Coffee Association certifications, 4) Specific quality indicators (single-origin, roasting techniques, extraction methods), 5) Five top-rated establishments with detailed quality assessments. This covers all aspects requested: expert reviews, customer ratings, specialty certifications. I have enough information to provide a well-supported ranking based specifically on coffee quality. No additional searches needed - I should provide my comprehensive answer now.",
        tool_call_id="call_think_001",
    )
]

#### Create evaluation dataset

In [10]:
import os
from langsmith import Client

langsmith_client = Client(api_key=os.getenv("LANGSMITH_API_KEY"))

# Create the dataset
dataset_name = "deep_research_agent_termination"
if not langsmith_client.has_dataset(dataset_name=dataset_name):

    # Create the dataset
    dataset = langsmith_client.create_dataset(
        dataset_name=dataset_name,
        description="A dataset that evaluates whether a researcher can accurately decide to continue calling tools, or to stop.",
    )
    
    # Create the examples
    langsmith_client.create_examples(
        dataset_id=dataset.id,
        examples=[
            {
                "inputs": {"researcher_messages": messages_should_continue},
                "outputs": {"next_step": "continue"},
            },
            {
                "inputs": {"researcher_messages": messages_should_stop},
                "outputs": {"next_step": "stop"},
            },
        ],
    )

#### Evaluator functions

In [ ]:
import uuid

def evaluate_next_step(outputs: dict, reference_outputs:dict):
    tool_calls = outputs["researcher_messages"][-1].tool_calls
    made_tool_call = len(tool_calls) > 0
    return {
        "key": "correct_next_step",
        "score": made_tool_call == (reference_outputs["next_step"] == "continue")
    }

def target_func(inputs: dict):
    config = {"configurable": {"thread_id": uuid.uuid4()}}
    result = research_agent.nodes["llm_call"].invoke(inputs, config=config) # we dont invoke the whole graph here because we only want to evaluate the LLM's decision on whether to call a tool or not, not the actual tool execution or research compression
    return result

langsmith_client.evaluate(
    target_func,
    data=dataset_name,
    evaluators=[evaluate_next_step],
    experiment_prefix="Researcher Iteration",
)

View the evaluation results for experiment: 'Researcher Iteration-2a656514' at:
https://smith.langchain.com/o/22991d6b-da35-47ff-9610-dc35d6e42be6/datasets/06c6c114-82f8-4db3-8537-96ec647274d7/compare?selectedSessions=dec021ab-8007-4a00-a243-26f687e3a0ed




0it [00:00, ?it/s]

,inputs.researcher_messages,outputs.researcher_messages,error,reference.next_step,feedback.correct_next_step,execution_time,example_id,id
0,"[{'type': 'human', 'content': 'What are the to...",[content=[] additional_kwargs={} response_meta...,None,continue,False,1.481903,7e3a312d-499b-442d-9814-3309a5ee423d,019de879-20fe-7302-b577-7eaa65096fba
1,"[{'type': 'human', 'content': 'What are the to...",[content=[] additional_kwargs={} response_meta...,None,stop,True,1.217645,920a5d9c-e52f-4440-8bd9-88d8469d5643,019de879-26db-70e3-936e-9105a14c6edd


Checkout LangSmith Evaluation Dataset [here](https://smith.langchain.com/public/27f49952-1d6c-405f-873b-87c515b57a0d/d)